# Confidence calibration

How well the model's stated `confidence` matches how often it is right, for one `evaluate` sweep.

After https://arxiv.org/pdf/2512.16030

In [ ]:
%matplotlib ipympl

In [ ]:
import pandas as pd

from analysis.build_data import load_trace_df
from analysis.calibration import expected_calibration_error, kalshibench_metrics


In [26]:
# one row per forecast of every child run of the sweep (see analysis/build_data.py)
forecasts = load_trace_df("outcome-confidence", run_name="sweep-20261106-1744")

In [27]:
forecasts

,run,model,experiment,exchange,category,event_type,market_title,response,as_of,predicted_probability,confidence,predicted_resolution,market_probability,outcome_occurred,resolution,correct,state
0,claude-fable-5-1-outcome-confidence,claude-fable-5-1,outcome-confidence,None,None,None,Will the Liberal Democratic Party of Russia (L...,None,2026-09-03 00:00:25,None,NaN,None,NaN,NaN,None,<NA>,ERROR
1,claude-fable-5-1-outcome-confidence,claude-fable-5-1,outcome-confidence,None,None,None,"Will Trump say ""Hell"" 10+ times during RNC Con...",None,2026-09-04 00:00:18,None,NaN,None,NaN,NaN,None,<NA>,ERROR
2,claude-fable-5-1-outcome-confidence,claude-fable-5-1,outcome-confidence,None,None,None,Will average **gas prices** be above $4.1900?,None,2026-10-25 04:00:00,None,NaN,None,NaN,NaN,None,<NA>,ERROR
3,claude-fable-5-1-outcome-confidence,claude-fable-5-1,outcome-confidence,None,None,None,Will average **gas prices** be above $3.7400?,None,2026-10-28 04:00:00,None,NaN,None,NaN,NaN,None,<NA>,ERROR
4,claude-fable-5-1-outcome-confidence,claude-fable-5-1,outcome-confidence,None,None,None,Will White House post 100-119 posts from July ...,None,2026-07-26 00:00:15,None,NaN,None,NaN,NaN,None,<NA>,ERROR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2393,claude-opus-4-5-outcome-confidence,claude-opus-4-5,outcome-confidence,polymarket,politics,speech quote,"Will Trump say ""Tax"" 7+ times during Rose Gard...","Looking at this prediction market, I need to a...",2026-06-26 00:00:03,None,0.55,No,0.9995,1.0,Yes,False,OK
2394,claude-opus-4-5-outcome-confidence,claude-opus-4-5,outcome-confidence,kalshi,economics,None,Will the rate of core PCE inflation be above 0...,Looking at this market about core PCE inflatio...,2025-09-12 04:00:00,None,0.98,Yes,0.9900,1.0,yes,True,OK
2395,claude-opus-4-5-outcome-confidence,claude-opus-4-5,outcome-confidence,polymarket,politics,social media post,Will NYC Mayor post 20-39 posts from July 24 t...,Looking at this prediction market question:\n\...,2026-07-22 00:00:12,None,0.55,No,0.2150,1.0,Yes,False,OK
2396,claude-opus-4-5-outcome-confidence,claude-opus-4-5,outcome-confidence,polymarket,politics,speech quote,Trump announces US x Iran ceasefire end by Apr...,"Looking at this market, I need to assess wheth...",2026-04-09 07:00:00,None,0.75,No,0.1850,0.0,No,True,OK


## Expected calibration error

ECE (Naeini et al., 2015) is the average gap between confidence and accuracy, with forecasts binned by their stated confidence into $B$ bins:

$$\mathrm{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{N}\,\bigl|\mathrm{acc}(B_b) - \mathrm{conf}(B_b)\bigr|$$

Here a forecast is the label the model predicts the market resolves to, `confidence` is the model's stated probability that the label is right, and a forecast is accurate when the label matches the market's `resolution` (ignoring case).

In [28]:
# scoreable forecasts: a predicted resolution with a confidence, on a market that has resolved
calibrated = forecasts.dropna(subset=["correct", "confidence"]).copy()

In [29]:
B = 10

In [30]:
confidence_bins = pd.IntervalIndex.from_breaks([b / B for b in range(B + 1)])

In [31]:
calibrated["confidence_bin"] = pd.cut(
    calibrated["confidence"], confidence_bins, include_lowest=True
)

In [32]:
ece = expected_calibration_error(calibrated)

In [33]:
ece_by_model = calibrated.groupby("model").apply(
    expected_calibration_error, include_groups=False
)

## KalshiBench table

Table 4 of [the paper](https://arxiv.org/pdf/2512.16030) reports, per model: accuracy, macro F1 and the F1 of the "yes" class, the Brier score of $P(\text{yes})$ against the outcome, the Brier skill score against a forecast of the base rate, $\mathrm{BSS} = 1 - \mathrm{BS} / \bar{y}(1 - \bar{y})$, and ECE and MCE (the worst bin's gap, $\max_b |\mathrm{acc}(B_b) - \mathrm{conf}(B_b)|$).

Here "yes" means the priced outcome occurs (`outcome_occurred`): the model said yes when its predicted resolution was that outcome, and its $P(\text{yes})$ is its confidence when it said yes and $1 -$ confidence otherwise.

In [34]:
# the model called the priced outcome: its label was right on a market where that outcome occurred, or wrong where it didn't
calibrated["predicted_yes"] = calibrated["correct"] == calibrated[
    "outcome_occurred"
].astype(bool)

In [35]:
calibrated["p_yes"] = calibrated["confidence"].where(
    calibrated["predicted_yes"], 1 - calibrated["confidence"]
)

In [36]:
model_table = calibrated.groupby("model").apply(
    kalshibench_metrics, include_groups=False
)

In [37]:
model_table.round(3)

,Acc,F1,F1yes,Brier,BSS,ECE,MCE
model,,,,,,,
claude-fable-5-1,72.222,0.632,0.451,0.180,0.186,0.041,0.167
claude-opus-4-5,69.208,0.628,0.473,0.210,0.038,0.046,0.350
claude-opus-5-5,72.163,0.631,0.449,0.184,0.177,0.042,0.091


In [38]:
# the reliability table behind the number: per bin, its size, conf(B_b) and acc(B_b)
reliability = calibrated.groupby("confidence_bin", observed=True).agg(
    n=("correct", "size"), conf=("confidence", "mean"), acc=("correct", "mean")
)